In [ ]:
!pip install -U -q accelerate transformers
!pip install -U -q bitsandbytes>=0.46.1
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 80.7 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import urllib.request
import urllib.parse
from collections import defaultdict, deque
import pandas as pd
import torch
from transformers import (AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,)
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize
import fitz

In [ ]:
ALLOWED_RELATIONS = [
    "uses","exploita", "targets", "exploits", "attributed-to", "downloads",
    "authored-by", "variant-of", "communicates-with", "delivers",
    "beacons-to", "consists-of", "hosts", "impersonates",
    "exfiltrates-to", "drops", "controls", "compromises",
    "originates-from", "owns", "indicates", "based-on",
    "duplicate-of", "related-to", "located-at", "has",
]

ALLOWED_ENTITIES = sorted([
    "threat-actor", "malware", "tools", "SOFTWARE", "vulnerability",
    "identity", "location", "url", "IPV4", "Infrastucture",
    "attack-pattern", "campaign", "FILEPATH", "REGISTRYKEY",
    "hash", "EMAIL", "TIME"
])

# CTI keyword filter — sentence must contain at least 2
CTI_KEYWORDS = {
    "malware", "apt", "actor", "exploit", "cve", "attack", "campaign",
    "backdoor", "trojan", "ransomware", "phishing", "c2", "payload",
    "vulnerability", "deploy", "download", "beacon", "exfiltrate",
    "spear", "lateral", "persistence", "credential", "privilege",
}

def is_cti_relevant(sentence: str) -> bool:
    lower = sentence.lower()
    return sum(1 for kw in CTI_KEYWORDS if kw in lower) >= 2


In [ ]:
# getting the apt_repos and 25 files
def get_repo(owner="blackorbird", repo="APT_REPORT",
                  branch="master") -> dict:
    url = (f"https://api.github.com/repos/{owner}/{repo}"
           f"/git/trees/{branch}?recursive=1")
    with urllib.request.urlopen(url) as resp:
        return json.load(resp)

def _norm(s: str) -> str:
    return s.lower().replace(" ", "").replace("-", "").replace("_", "")

def select_files(tree: dict, max_files: int = 25,max_file_size_mb: float = 8) -> list:
    THREAT_ACTORS = {
        "apt28", "apt29", "apt32", "apt33", "apt34", "apt38", "apt41",
        "kimsuky", "lazarus", "sandworm", "gamaredon", "turla",
        "fin7", "fin8", "darkside", "revil", "conti", "muddywater",
    }
    EXCLUDE = {"summary", "cybercrime", "aisecurity", "ot",
               "international strategic"}

    blobs = [
        t for t in tree["tree"]
        if t["type"] == "blob"
        and t["path"].lower().endswith((".pdf", ".txt", ".md"))
    ]

    by_folder: dict = defaultdict(list)
    for f in blobs:
        folder = f["path"].split("/")[0]
        if folder.lower() in EXCLUDE:
            continue
        if f["path"].split("/")[-1].lower() in ("readme.md", "readme.txt"):
            continue
        size_mb = f.get("size", 0) / 1e6
        if size_mb > max_file_size_mb:
            continue
        if _norm(folder) in {_norm(a) for a in THREAT_ACTORS}:
            by_folder[folder].append(f)

    # Round-robin so we get files from many groups, not just one
    queues = {
        k: deque(sorted(v, key=lambda x: x.get("size", 0)))
        for k, v in by_folder.items()
    }
    order = deque(queues.keys())
    selected = []
    while order and len(selected) < max_files:
        folder = order.popleft()
        q = queues[folder]
        if q:
            selected.append(q.popleft())
            if q:
                order.append(folder)

    groups = sorted({f["path"].split("/")[0] for f in selected})
    print(f"Selected {len(selected)} files across {len(groups)} APT groups:")
    print(" ", groups)
    return selected

def download_files(selected: list,
                   out_dir: str = "apt_subset",
                   owner: str = "blackorbird",
                   repo: str = "APT_REPORT",
                   branch: str = "master") -> list:
    os.makedirs(out_dir, exist_ok=True)
    downloaded = []
    for i, f in enumerate(selected, 1):
        path = f["path"]
        url = (f"https://raw.githubusercontent.com/{owner}/{repo}"
               f"/{branch}/" + urllib.parse.quote(path))
        local = os.path.join(out_dir, path.replace("/", "__"))
        try:
            urllib.request.urlretrieve(url, local)
            downloaded.append(local)
        except Exception as e:
            print(f"  skipped {path}: {e}")
        if i % 10 == 0 or i == len(selected):
            print(f"  downloaded {i}/{len(selected)}")
    return downloaded


print("Fetching file listing from GitHub")
tree = get_repo()
selected = select_files(tree, max_files=25)
report_files = download_files(selected)
print(f"\nReady: {len(report_files)} files downloaded.")


Fetching file listing from GitHub...
Selected 25 files across 10 APT groups:
  ['APT28', 'APT29', 'APT34', 'APT41', 'Gamaredon', 'Sandworm', 'Turla', 'kimsuky', 'lazarus', 'muddywater']
  downloaded 10/25
  downloaded 20/25
  downloaded 25/25

Ready: 25 files downloaded.


In [ ]:
#  Extracting text and building the sentence pool
def extract_text(path: str, max_pages: int = 8) -> str:
    # Extract plain text from PDF (first 8 pages) or text file.
    if path.lower().endswith(".pdf"):
        try:
            doc = fitz.open(path)
            pages = doc[:max_pages] if len(doc) > max_pages else doc
            return "\n".join(page.get_text() for page in pages)
        except Exception as e:
            print(f"  PDF skipped ({path}): {e}")
            return ""
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        return fh.read()

def build_sentence_pool(files: list, target: int = 500) -> list:
    # Tokenise every file into sentences, apply the CTI relevance
    # filter, deduplicate, and return up to `target` sentences.

    pool: set = set()
    for path in files:
        if len(pool) >= target:
            break
        text = extract_text(path)
        if not text.strip():
            continue
        cleaned = re.sub(r"\s+", " ", text)
        for sent in sent_tokenize(cleaned):
            sent = sent.strip()
            # length guard: not too short (noise) or too long (paragraphs)
            if 40 < len(sent) < 400 and is_cti_relevant(sent):
                pool.add(sent)
                if len(pool) >= target:
                    break

    sentences = list(pool)
    print(f"\nSentence pool: {len(sentences)} CTI-relevant sentences "
          f"(from {len(files)} files)")
    return sentences

sentences = build_sentence_pool(report_files, target=500)



Sentence pool: 222 CTI-relevant sentences (from 25 files)


In [ ]:
# Loading the Msitral LLM model
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL_ID} in 4-bit")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_cfg,
    device_map="auto",
    attn_implementation="sdpa",
)
print("Model loaded")


Loading mistralai/Mistral-7B-Instruct-v0.3 in 4-bit


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded


In [ ]:
# Prompt building and running it in the batch of 8
def build_prompt(sentence: str) -> str:
    """
    Single prompt template. Lists allowed relation types and entity
    types once, then asks for a JSON array of triples.
    apply_chat_template() handles Mistral's [INST] format automatically.
    """
    relations_str = ", ".join(ALLOWED_RELATIONS)
    entities_str = ", ".join(ALLOWED_ENTITIES)

    content = f"""You are a Cyber Threat Intelligence extraction engine.

Allowed relation types : {relations_str}
Allowed entity types   : {entities_str}

Extract every relationship triple from the sentence below.
Output ONLY a valid JSON array — no explanation, no markdown:
[{{"head": "...", "head_type": "...", "relation": "...", "tail": "...", "tail_type": "..."}}]

If no relation applies, output exactly: []

NOTE: spell Infrastructure as "Infrastucture" (matches the dataset label).

Sentence: "{sentence}"
Output:"""

    messages = [{"role": "user", "content": content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_candidates(
    sentences: list, batch_size: int = 8, max_new_tokens: int = 128
) -> pd.DataFrame:
    """
    Sends sentences to Mistral in batches and parses the JSON output.
    Returns a DataFrame with one row per extracted triple.
    """
    rows = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start : start + batch_size]
        prompts = [build_prompt(s) for s in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # greedy — deterministic
                pad_token_id=tokenizer.pad_token_id,
            )

        input_len = inputs["input_ids"].shape[1]

        for i, sentence in enumerate(batch):
            gen_text = tokenizer.decode(
                outputs[i][input_len:], skip_special_tokens=True
            )

            # Extract JSON array from model output
            match = re.search(
                r"\[\s*\{.*?\}\s*\]|\[\]", gen_text, re.DOTALL
            )
            if not match:
                continue
            try:
                triples = json.loads(match.group(0))
            except json.JSONDecodeError:
                continue

            for triple in triples:
                if isinstance(triple, dict):
                    triple["sentence"] = sentence
                    rows.append(triple)

        done = start + len(batch)
        if done % 32 == 0 or done == len(sentences):
            print(
                f"  generated {done}/{len(sentences)}"
                f"  ({len(rows)} candidates so far)"
            )

    df = pd.DataFrame(rows)
    print(f"\nTotal raw candidates extracted: {len(df)}")
    return df



print(
    f"Running generation on {len(sentences)} sentences "
    f"(batch_size=8, max_new_tokens=128)...\n"
)
raw_candidates_df = generate_candidates(sentences)

Running generation on 222 sentences (batch_size=8, max_new_tokens=128)...

  generated 32/222  (22 candidates so far)
  generated 64/222  (48 candidates so far)
  generated 96/222  (75 candidates so far)
  generated 128/222  (102 candidates so far)
  generated 160/222  (128 candidates so far)
  generated 192/222  (158 candidates so far)
  generated 222/222  (185 candidates so far)

Total raw candidates extracted: 185


In [ ]:
print("=== Sample candidates ===")
if not raw_candidates_df.empty:
    cols = ["head", "head_type", "relation", "tail", "tail_type"]
    print(raw_candidates_df[cols].head(20).to_string(index=False))
    print("\nRelation distribution:")
    print(raw_candidates_df["relation"].value_counts().to_string())
    print("\nHead type distribution:")
    print(raw_candidates_df["head_type"].value_counts().to_string())
else:
    print("No candidates extracted.")

# Save to JSON so you have the raw output regardless of what comes next
raw_candidates_df.to_json(
    "raw_llm_candidates.json", orient="records", indent=2
)
print("\nSaved to raw_llm_candidates.json")

=== Sample candidates ===
                                                                        head      head_type               relation                                      tail      tail_type
                                                                 Mail server  Infrastucture                targets                   administrator mailboxes          EMAIL
                                                        compromised accounts         threat              indicates                             organisations       identity
                                                                         IPs           IPV4                    has                         C2 infrastructure  Infrastucture
                                                                         IPs           IPV4                targets Sandworm-attributed Cyclops Blink malware        malware
                                                                  Armageddon   threat-actor                   uses

In [12]:
import os

print("Exact file path:")
print(os.path.abspath("raw_llm_candidates.json"))

Exact file path:
/content/raw_llm_candidates.json


In [13]:
from google.colab import files

files.download('/content/raw_llm_candidates.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>